In [6]:
import os
import importlib
import sys
from pathlib import Path
import json
from typing import Callable

from sklearn.utils import resample

from tqdm import tqdm
import numpy as np
import h5py

In [7]:
sys.path.append("../src")
import emd_loader
from cca import CCA_Classifier, evaluate_cca

import helpers
from helpers import save_features, load_features

In [8]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


In [9]:
# TODO: remove the EMD when starting the hackerthon
eeg_feature_file = Path("/content/drive/MyDrive/eeg_features/emd_biot/features.h5")
#eeg_feature_file = Path("/content/drive/MyDrive/eeg_features/emd_luna/features.h5")
#eeg_feature_file = Path("/content/drive/MyDrive/eeg_features/emd_labram/features.h5")
#eeg_feature_file = Path("/content/drive/MyDrive/eeg_features/emd_cbramod/features.h5")

video_feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_video_openclip")
#video_feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_video_dino")
#video_feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_video_videomae")

X, pre_Z = load_features(eeg_feature_file)
X, pre_Z = resample(X, pre_Z, n_samples=round(0.3 * len(X)), replace=False, stratify=pre_Z, random_state=1337)

In [12]:
num_videos = 1102
video_ids = range(1, num_videos+1)
"""
reference_video, _ = load_features(video_feature_dir / "feature-1.h5")
Z = np.empty((pre_Z.shape[0], reference_video.shape[0]))
d = {
    vid:load_features(video_feature_dir / f"feature-{vid}.h5")
    for vid in tqdm(range(1, num_videos+1), desc="Load features")
}
for i in tqdm(range(pre_Z.shape[0]), desc="Construct Y"):
    Z[i], _ = d[pre_Z[i]]
"""
video_features = np.stack([
    load_features(video_feature_dir / f"feature-{vid}.h5")[0].ravel()
    for vid in tqdm(video_ids, desc="Load features")
])

Load features: 100%|██████████| 1102/1102 [00:04<00:00, 255.77it/s]


In [13]:
evaluate_cca(X, pre_Z, video_ids, video_features)

{'n_components': 2}: CV accuracy 0.0008
Best parameters: {'n_components': 2}
Best CV accuracy: 0.0007780587728737323
Test accuracy: 0.0
Chance level: 0.0009074410163339383


{'n_components': 2}